<a href="https://colab.research.google.com/github/DJ-Ortega/unsupervised-trading-strat/blob/main/Unsupervised_strat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unsupervised Learning Trading Strategy

* Download/Load SP500 stocks prices data.
* Calculate different features and indicators on each stock.
* Aggregate on monthly level and filter top 150 most liquid stocks.
* Calculate Monthly Returns for different time-horizons.
* Download Fama-French Factors and Calculate Rolling Factor Betas.
* For each month fit a K-Means Clustering Algorithm to group similar assets based on their features.
* For each month select assets based on the cluster and form a portfolio based on Efficient Frontier max sharpe ratio optimization.
* Visualize Portfolio returns and compare to SP500 returns.

Be mindful that this backtest does not take in survivorship bias

In [1]:
!pip install pandas-datareader --quiet
!pip install pyportfolioopt --quiet
!pip install pandas_ta --quiet
!pip install --upgrade yfinance --quiet
!pip install pandas==2.3.3 --quiet
!pip install numpy==2.2.6 --quiet

In [2]:
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta
import warnings
warnings.filterwarnings('ignore')

## 1. Download/Load SP500 stocks prices data.

In [3]:
import requests
from io import StringIO

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (student learning project)'}
response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))
sp500 = tables[0]
sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')
symbols_list = sp500['Symbol'].unique().tolist()

end_date = '2026-10-30'
start_date = pd.to_datetime(end_date) - pd.DateOffset(365*8)

raw_df = yf.download(tickers=symbols_list,
                     start=start_date,
                     end=end_date)

[*********************100%***********************]  503 of 503 completed


In [4]:
#cleaning the dataframe
df = raw_df.stack()
df.index.names = ['date', 'ticker']
df.columns = df.columns.str.lower()
df['adj close'] = df['close']
df

Price                   close        high         low        open  \
date       ticker                                                   
2018-11-01 A        61.491665   61.604800   60.888251   61.350238   
           AAPL     52.496853   52.529926   51.218803   51.747978   
           ABBV     57.475525   57.790814   55.533640   55.684117   
           ABT      60.562424   60.735087   59.345129   59.647292   
           ACGL     27.281279   27.576057   26.739267   27.119626   
...                       ...         ...         ...         ...   
2026-10-29 XYZ      76.949997   77.559998   74.809998   74.830002   
           YUM     154.550003  154.750000  152.000000  152.520004   
           ZBH      97.470001   97.989998   96.260002   97.330002   
           ZBRA    377.380005  389.230011  374.549988  384.929993   
           ZTS      78.270004   78.746772   77.236998   77.912425   

Price                   volume   adj close  
date       ticker                           
2018-11-01 A         1882400.0   61.491665  
           AAPL    233292800.0   52.496853  
           ABBV     10671700.0   57.475525  
           ABT       6250800.0   60.562424  
           ACGL      2081000.0   27.281279  
...                        ...         ...  
2026-10-29 XYZ       4331600.0   76.949997  
           YUM       1606700.0  154.550003  
           ZBH       2040800.0   97.470001  
           ZBRA       527500.0  377.380005  
           ZTS       2935500.0   78.270004  

[988237 rows x 6 columns]

## 2. Calculate features and technical indicators for each stock.

* Garman-Klass Volatility
* RSI
* Bollinger Bands
* ATR
* MACD
* Dollar Volume

\begin{equation}
\text{Garman-Klass Volatility} = \frac{(\ln(\text{High}) - \ln(\text{Low}))^2}{2} - (2\ln(2) - 1)(\ln(\text{Adj Close}) - \ln(\text{Open}))^2
\end{equation}

$$z = \frac{\text{ATR} - \text{mean}}{\text{std}}$$

In [5]:
df['garman_klass_vol'] = ((np.log(df['high'])-np.log(df['low']))**2)/2 - (2*np.log(2)-1)*(np.log(df['adj close'])-np.log(df['open']))**2

df['rsi'] = df.groupby(level = 1)['adj close'].transform(lambda x: pandas_ta.rsi(close = x, length = 20))
#df.xs('AAPL', level=1)['rsi'].plot()

df['bb_low'] = df.groupby(level = 1)['adj close'].transform(lambda x: pandas_ta.bbands(close = np.log1p(x), length = 20).iloc[:,0])

df['bb_mid'] = df.groupby(level = 1)['adj close'].transform(lambda x: pandas_ta.bbands(close = np.log1p(x), length = 20).iloc[:,1])

df['bb_high'] = df.groupby(level = 1)['adj close'].transform(lambda x: pandas_ta.bbands(close = np.log1p(x), length = 20).iloc[:,2])

# atr function
def compute_atr(stock_data):
  atr = pandas_ta.atr(high = stock_data['high'],
                      low = stock_data['low'],
                      close = stock_data['adj close'],
                      length = 14)
  return atr.sub(atr.mean()).div(atr.std())

df['atr'] = df.groupby(level = 1, group_keys = False).apply(compute_atr)

# macd function
def compute_macd(close):
    macd = pandas_ta.macd(close=close)
    if macd is None:
        return pd.Series(np.nan, index=close.index)
    macd = macd.iloc[:,0]
    return macd.sub(macd.mean()).div(macd.std())

df['macd'] = df.groupby(level = 1, group_keys = False)['adj close'].apply(compute_macd)

df['dollar_volume'] = (df['adj close']*df['volume'])/1e6 #divide by million

df


Price                   close        high         low        open  \
date       ticker                                                   
2018-11-01 A        61.491665   61.604800   60.888251   61.350238   
           AAPL     52.496853   52.529926   51.218803   51.747978   
           ABBV     57.475525   57.790814   55.533640   55.684117   
           ABT      60.562424   60.735087   59.345129   59.647292   
           ACGL     27.281279   27.576057   26.739267   27.119626   
...                       ...         ...         ...         ...   
2026-10-29 XYZ      76.949997   77.559998   74.809998   74.830002   
           YUM     154.550003  154.750000  152.000000  152.520004   
           ZBH      97.470001   97.989998   96.260002   97.330002   
           ZBRA    377.380005  389.230011  374.549988  384.929993   
           ZTS      78.270004   78.746772   77.236998   77.912425   

Price                   volume   adj close  garman_klass_vol        rsi  \
date       ticker                                                         
2018-11-01 A         1882400.0   61.491665          0.000066        NaN   
           AAPL    233292800.0   52.496853          0.000240        NaN   
           ABBV     10671700.0   57.475525          0.000406        NaN   
           ABT       6250800.0   60.562424          0.000178        NaN   
           ACGL      2081000.0   27.281279          0.000461        NaN   
...                        ...         ...               ...        ...   
2026-10-29 XYZ       4331600.0   76.949997          0.000350  49.542236   
           YUM       1606700.0  154.550003          0.000093  61.320330   
           ZBH       2040800.0   97.470001          0.000158  56.610909   
           ZBRA       527500.0  377.380005          0.000587  53.726565   
           ZTS       2935500.0   78.270004          0.000179  56.769603   

Price                bb_low    bb_mid   bb_high       atr      macd  \
date       ticker                                                     
2018-11-01 A            NaN       NaN       NaN       NaN       NaN   
           AAPL         NaN       NaN       NaN       NaN       NaN   
           ABBV         NaN       NaN       NaN       NaN       NaN   
           ABT          NaN       NaN       NaN       NaN       NaN   
           ACGL         NaN       NaN       NaN       NaN       NaN   
...                     ...       ...       ...       ...       ...   
2026-10-29 XYZ     4.314677  4.352530  4.390383 -0.763525 -0.078185   
           YUM     4.924020  4.997748  5.071475  1.515138  1.373804   
           ZBH     4.475066  4.549015  4.622964 -0.477666  0.560336   
           ZBRA    5.918707  5.947949  5.977191  0.369165  0.428429   
           ZTS     4.246431  4.344412  4.442392 -1.572758  0.631972   

Price              dollar_volume  
date       ticker                 
2018-11-01 A          115.751910  
           AAPL     12247.137798  
           ABBV       613.361559  
           ABT        378.563598  
           ACGL        56.772341  
...                          ...  
2026-10-29 XYZ        333.316607  
           YUM        248.315490  
           ZBH        198.916778  
           ZBRA       199.067953  
           ZTS        229.761598  

[988237 rows x 14 columns]

## 3. Aggregate to monthly level and filter top 150 most liquid stocks for each month.

* To reduce training time and experiment with features and strategies, we convert the business-daily data to month-end frequency.

In [29]:
last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open', 'high', 'low', 'close']]

data = (pd.concat([df.unstack('ticker')['dollar_volume'].resample('ME').mean().stack('ticker').to_frame('dollar_volume'),
                   df.unstack('ticker')[last_cols].resample('ME').last().stack('ticker')],
                  axis=1)).dropna()

data['dollar_volume'] = data['dollar_volume'].unstack('ticker').rolling(5*12).mean().stack('ticker')

data['dollar_vol_rank'] = data.groupby('date')['dollar_volume'].rank(ascending=False)

data = data[data['dollar_vol_rank'] < 150].drop(['dollar_volume', 'dollar_vol_rank'], axis=1)
data

adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2023-11-30 AAPL    187.662109          0.000063  62.715645  5.168431   
           ABBV    128.870636          0.000181  51.608067  4.820542   
           ABT      97.914436          0.000110  66.470043  4.466437   
           ACN     314.206970          0.000032  65.329792  5.671690   
           ADBE    611.010010          0.000276  62.481317  6.327316   
...                       ...               ...        ...       ...   
2026-10-31 WDC     395.410004          0.000550  47.714593  5.797405   
           WFC      84.309937          0.000088  53.894828  4.387687   
           WMT     121.250000          0.000139  65.689481  4.655408   
           XOM     183.289993          0.000176  67.139510  5.088871   
           XYZ      76.949997          0.000350  49.542236  4.314677   

                     bb_mid   bb_high       atr      macd  
date       ticker                                          
2023-11-30 AAPL    5.220889  5.273347 -0.632218  0.839238  
           ABBV    4.846052  4.871561 -0.481843 -0.780736  
           ABT     4.540050  4.613664 -0.651737  1.241512  
           ACN     5.724886  5.778082 -0.703406  1.203849  
           ADBE    6.394875  6.462435  0.486640  1.793385  
...                     ...       ...       ...       ...  
2026-10-31 WDC     5.941118  6.084832  1.986838 -2.174089  
           WFC     4.416764  4.445841  1.153671 -0.219729  
           WMT     4.752395  4.849383  1.691367  2.967297  
           XOM     5.168683  5.248495  2.146448  2.904047  
           XYZ     4.352530  4.390383 -0.763525 -0.078185  

[5364 rows x 8 columns]